In [ ]:
import os
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")                                                       
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                             adjusted_rand_score, rand_score)
from sklearn.metrics.pairwise import pairwise_distances

warnings.filterwarnings("ignore")

                                                                      
                                                     
                                                                      
CSV_PATH      = "covtype.csv"
OUTDIR        = "results"
SAMPLE_SIZE   = 20000                                                         
N_PCA         = 10                                      
SIL_SAMPLE    = 10000                                             
DUNN_SAMPLE   = 2500                                            
RANDOM_STATE  = 42

K_VALUES      = [2, 3, 4, 5, 6, 7, 8, 9, 10]
EPS_VALUES    = [0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.5]
MIN_SAMPLES   = [5, 10, 20, 50]

os.makedirs(OUTDIR, exist_ok=True)
rng = np.random.RandomState(RANDOM_STATE)

CONTINUOUS = ["Elevation", "Aspect", "Slope",
              "Horizontal_Distance_To_Hydrology", "Vertical_Distance_To_Hydrology",
              "Horizontal_Distance_To_Roadways", "Hillshade_9am", "Hillshade_Noon",
              "Hillshade_3pm", "Horizontal_Distance_To_Fire_Points"]


                                                                      
                                             
                                                                      
print("[1] Loading data ...")
df = pd.read_csv(CSV_PATH)
print("    full shape:", df.shape)

y_full = df["Cover_Type"].values                                        
X_full = df.drop(columns=["Cover_Type"])                                   


                                                                      
                            
                                                                      
print("[2] Stratified sampling to", SAMPLE_SIZE, "rows ...")
X, _, y, _ = train_test_split(X_full, y_full,
                              train_size=SAMPLE_SIZE,
                              stratify=y_full,
                              random_state=RANDOM_STATE)
X = X.reset_index(drop=True)
print("    class distribution in sample:")
print(pd.Series(y).value_counts().sort_index().to_string())


                                                                      
                          
                                                                      
print("[3] Scaling continuous features and applying PCA ...")
Xs = X.astype(float).copy()
Xs[CONTINUOUS] = StandardScaler().fit_transform(Xs[CONTINUOUS])
Xs = Xs.values

pca = PCA(n_components=N_PCA, random_state=RANDOM_STATE)
Xr = pca.fit_transform(Xs)
print(f"    variance retained by {N_PCA} components: "
      f"{pca.explained_variance_ratio_.sum():.3f}")

X2 = Xr[:, :2]                                                            


                                                                      
                             
                                                                      
def dunn_index(X, labels, max_points=DUNN_SAMPLE, random_state=0):
    m = labels != -1
    X, labels = X[m], labels[m]
    if len(np.unique(labels)) < 2:
        return np.nan
    if len(X) > max_points:
        r = np.random.RandomState(random_state)
        sub = r.choice(len(X), max_points, replace=False)
        X, labels = X[sub], labels[sub]
    D = pairwise_distances(X)
    uniq = np.unique(labels)
    max_diam, min_sep = 0.0, np.inf
    for i, a in enumerate(uniq):
        ia = labels == a
        if ia.sum() > 1:
            max_diam = max(max_diam, D[np.ix_(ia, ia)].max())
        for b in uniq[i + 1:]:
            ib = labels == b
            d = D[np.ix_(ia, ib)]
            if d.size:
                min_sep = min(min_sep, d.min())
    return min_sep / max_diam if max_diam > 0 else np.nan


def evaluate(X, labels, y_true):
    out = {}
    k = len(set(labels) - {-1})
    out["n_clusters"] = k
    out["n_noise"] = int((labels == -1).sum())
    out["noise_pct"] = 100.0 * out["n_noise"] / len(labels)

    if k < 2:
        out.update(silhouette=np.nan, davies_bouldin=np.nan, dunn=np.nan,
                   rand=np.nan, adj_rand=np.nan)
        return out

    m = labels != -1
    out["silhouette"] = silhouette_score(
        X[m], labels[m],
        sample_size=min(SIL_SAMPLE, m.sum()), random_state=RANDOM_STATE)
    out["davies_bouldin"] = davies_bouldin_score(X[m], labels[m])
    out["dunn"] = dunn_index(X, labels, random_state=RANDOM_STATE)
    out["rand"] = rand_score(y_true, labels)
    out["adj_rand"] = adjusted_rand_score(y_true, labels)
    return out


                                                                      
                                                           
                                                                      
print("[4] Building k-distance plot for eps selection ...")
k_nn = 10
nn = NearestNeighbors(n_neighbors=k_nn).fit(Xr)
dist, _ = nn.kneighbors(Xr)
kdist = np.sort(dist[:, -1])

plt.figure(figsize=(7, 4.5))
plt.plot(kdist, lw=1.6)
for e in EPS_VALUES:
    plt.axhline(e, ls="--", lw=0.7, color="grey")
plt.xlabel("Points sorted by distance")
plt.ylabel(f"Distance to {k_nn}-th nearest neighbour")
plt.title("k-distance plot — the knee suggests a good eps")
plt.grid(alpha=.3); plt.tight_layout()
plt.savefig(f"{OUTDIR}/fig1_kdistance.png", dpi=150); plt.close()
print(f"    k-dist percentiles  50%={np.percentile(kdist,50):.2f}  "
      f"90%={np.percentile(kdist,90):.2f}  95%={np.percentile(kdist,95):.2f}")


                                                                      
                                  
                                                                      
print("[5] K-Means sweep ...")
km_rows, km_models = [], {}
for k in K_VALUES:
    t0 = time.time()
    km = KMeans(n_clusters=k, n_init=10, init="k-means++",
                random_state=RANDOM_STATE).fit(Xr)
    row = {"k": k, "inertia": km.inertia_, "runtime_s": round(time.time() - t0, 2)}
    row.update(evaluate(Xr, km.labels_, y))
    km_rows.append(row)
    km_models[k] = km.labels_
    print(f"    k={k:<3} sil={row['silhouette']:.3f}  DB={row['davies_bouldin']:.3f}"
          f"  dunn={row['dunn']:.4f}  ARI={row['adj_rand']:.4f}")

km_df = pd.DataFrame(km_rows)
km_df.to_csv(f"{OUTDIR}/kmeans_results.csv", index=False)


                                                                      
                                 
                                                                      
print("[6] DBSCAN sweep ...")
db_rows, db_models = [], {}
for eps in EPS_VALUES:
    for ms in MIN_SAMPLES:
        t0 = time.time()
        lab = DBSCAN(eps=eps, min_samples=ms, n_jobs=-1).fit_predict(Xr)
        row = {"eps": eps, "min_samples": ms, "runtime_s": round(time.time() - t0, 2)}
        row.update(evaluate(Xr, lab, y))
        db_rows.append(row)
        db_models[(eps, ms)] = lab
        print(f"    eps={eps:<5} ms={ms:<3} clusters={row['n_clusters']:<3}"
              f" noise={row['noise_pct']:5.1f}%  sil={row['silhouette']:.3f}"
              f"  ARI={row['adj_rand']:.4f}")

db_df = pd.DataFrame(db_rows)
db_df.to_csv(f"{OUTDIR}/dbscan_results.csv", index=False)


                                                                      
                 
                                                                      
print("[7] Drawing figures ...")

                                                                            
fig, ax = plt.subplots(2, 3, figsize=(15, 8))
panels = [("inertia", "Inertia (elbow)", "lower = tighter"),
          ("silhouette", "Silhouette", "higher better"),
          ("davies_bouldin", "Davies-Bouldin", "lower better"),
          ("dunn", "Dunn", "higher better"),
          ("adj_rand", "Adjusted Rand", "higher better"),
          ("rand", "Rand Index", "higher better")]
for a, (col, title, note) in zip(ax.ravel(), panels):
    a.plot(km_df["k"], km_df[col], "o-")
    a.set_title(f"{title}  ({note})"); a.set_xlabel("k"); a.grid(alpha=.3)
fig.suptitle("K-Means: validation indices vs number of clusters", y=1.01)
fig.tight_layout(); fig.savefig(f"{OUTDIR}/fig2_kmeans_sweep.png", dpi=150,
                                bbox_inches="tight"); plt.close(fig)

                                                                           
metrics = ["n_clusters", "noise_pct", "silhouette", "davies_bouldin",
           "dunn", "adj_rand"]
fig, ax = plt.subplots(2, 3, figsize=(16, 8))
for a, met in zip(ax.ravel(), metrics):
    piv = db_df.pivot(index="min_samples", columns="eps", values=met)
    im = a.imshow(piv.values, cmap="viridis", aspect="auto")
    a.set_xticks(range(len(piv.columns))); a.set_xticklabels(piv.columns)
    a.set_yticks(range(len(piv.index)));  a.set_yticklabels(piv.index)
    a.set_xlabel("eps"); a.set_ylabel("min_samples"); a.set_title(met)
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            v = piv.values[i, j]
            if not np.isnan(v):
                a.text(j, i, f"{v:.2f}", ha="center", va="center",
                       color="w", fontsize=7)
    fig.colorbar(im, ax=a, fraction=.046)
fig.suptitle("DBSCAN: metrics across the (eps, min_samples) grid", y=1.01)
fig.tight_layout(); fig.savefig(f"{OUTDIR}/fig3_dbscan_heatmaps.png", dpi=150,
                                bbox_inches="tight"); plt.close(fig)

                                                                           
best_km_k = int(km_df.loc[km_df["silhouette"].idxmax(), "k"])

valid = db_df[(db_df["n_clusters"] >= 2) & (db_df["noise_pct"] < 30)]
if valid.empty:
    valid = db_df[db_df["n_clusters"] >= 2]
best_db = valid.loc[valid["silhouette"].idxmax()]
best_db_key = (best_db["eps"], int(best_db["min_samples"]))

                                                                            
fig, ax = plt.subplots(1, 3, figsize=(16, 5))
ax[0].scatter(X2[:, 0], X2[:, 1], c=y, s=3, cmap="tab10")
ax[0].set_title("True Cover_Type (reference only)")
ax[1].scatter(X2[:, 0], X2[:, 1], c=km_models[best_km_k], s=3, cmap="tab10")
ax[1].set_title(f"K-Means, k={best_km_k}")
lab_db = db_models[best_db_key]
ax[2].scatter(X2[lab_db == -1, 0], X2[lab_db == -1, 1], c="lightgrey", s=3,
              label="noise")
ax[2].scatter(X2[lab_db != -1, 0], X2[lab_db != -1, 1], c=lab_db[lab_db != -1],
              s=3, cmap="tab10")
ax[2].set_title(f"DBSCAN, eps={best_db_key[0]}, min_samples={best_db_key[1]}")
ax[2].legend(markerscale=4, loc="best", fontsize=8)
for a in ax:
    a.set_xlabel("PC1"); a.set_ylabel("PC2")
fig.tight_layout(); fig.savefig(f"{OUTDIR}/fig4_pca_scatter.png", dpi=150)
plt.close(fig)

                                                                            
km_best = km_df[km_df["k"] == best_km_k].iloc[0]
comp = pd.DataFrame({
    "Metric": ["Silhouette", "Dunn", "Davies-Bouldin", "Rand", "Adjusted Rand"],
    f"K-Means (k={best_km_k})": [km_best.silhouette, km_best.dunn,
                                 km_best.davies_bouldin, km_best.rand,
                                 km_best.adj_rand],
    f"DBSCAN (eps={best_db_key[0]}, ms={best_db_key[1]})":
        [best_db.silhouette, best_db.dunn, best_db.davies_bouldin,
         best_db.rand, best_db.adj_rand]})
comp.to_csv(f"{OUTDIR}/best_comparison.csv", index=False)

x = np.arange(len(comp)); w = 0.35
plt.figure(figsize=(9, 5))
plt.bar(x - w/2, comp.iloc[:, 1], w, label=comp.columns[1])
plt.bar(x + w/2, comp.iloc[:, 2], w, label=comp.columns[2])
plt.xticks(x, comp["Metric"]); plt.ylabel("Index value")
plt.title("Best K-Means vs best DBSCAN")
plt.legend(); plt.grid(axis="y", alpha=.3); plt.tight_layout()
plt.savefig(f"{OUTDIR}/fig5_comparison.png", dpi=150); plt.close()


                                                                      
                          
                                                                      
print("\n================ K-MEANS RESULTS ================")
print(km_df.round(4).to_string(index=False))
print("\n================ DBSCAN RESULTS =================")
print(db_df.round(4).to_string(index=False))
print("\n================ BEST OF EACH ===================")
print(comp.round(4).to_string(index=False))
print(f"\nAll tables and figures saved in ./{OUTDIR}/")

: 